============================================================
# HTU INTELLIGENT REGISTRATION SYSTEM - FULL DATA GENERATOR
============================================================
This script does two things in sequence:
  1. Generates corrected CSV files for all 6 study plans
     (CS / AI / Cybersecurity  x  Bachelor / Technical)
     based directly on the official HTU PDFs (2022 plans).
  2. Simulates realistic student enrollment histories for
     3000 students, producing htu_students.csv and
     htu_enrollments.csv ready for ML training.

All business rules are enforced:
  - Correct semester order: Fall(Y) → Spring(Y+1) → Summer(Y+1)
  - Hard and co-requisite prerequisites (two-pass eligibility)
  - Credit limits:  regular 12-18h,  summer 3-9h
      graduation exception: up to 21h (12h in summer) if
      remaining hours fall below the normal minimum
  - Apprenticeship block: 2 consecutive semesters,
      no other courses allowed except Capstone
  - Independent remedial courses (each has its own exam)
  - Failed-course retake priority
  - Graduation detection per plan

Author: auto-generated from HTU official plan PDFs (2022)
Run:    python htu_full_generator.py
============================================================


## Imports and setup

In [1]:

import pandas as pd
import random
import os
import sys

random.seed(24)                  # Remove or change for different runs

# Where to write the output files (write inside a folder called "output" to avoid cluttering the current directory) 
OUT_DIR = os.path.abspath("output")




==============================================================
## PART A  —  COMPLETE PLAN DATA  (sourced from official PDFs)
==============================================================



### ── A1. SHARED COURSE CATALOG ─────────────────────────────────
Format: code → (name_en, credit_hours, lecture_hours, lab_hours)

Course codes use the full 10-digit format from the HTU system.

lecture_hours and lab_hours are *contact* hours per week.


### ── A2.  UNIVERSITY ELECTIVE POOL (same for all plans) ───────

In [2]:

# ── A1. SHARED COURSE CATALOG ─────────────────────────────────
# Format: code → (name_en, credit_hours, lecture_hours, lab_hours)
# Course codes use the full 10-digit format from the HTU system.
# lecture_hours and lab_hours are *contact* hours per week.

CATALOG = {
    # ── Remedial (0 credit hours – do NOT count toward graduation) ──
    "0030301110": ("Remedial Arabic Language",                  0, 3, 0),
    "0030301120": ("Pre-Foundation English Intensive + Lab",    0, 4, 0),
    "0030303110": ("Remedial Math",                             0, 2, 0),

    # ── University Compulsory ──────────────────────────────────────
    "0030301111": ("Arabic Language & Communication Skills",    1, 3, 0),
    "0030301121": ("English Pre-Intermediate Intensive + Lab",  4, 4, 0),
    "0030301122": ("English Intermediate",                      3, 4, 0),
    "0030301123": ("English Upper-Intermediate",                3, 4, 0),
    "0030301124": ("English Advanced",                          3, 4, 0),  # BSc only
    "0040302111": ("Professional Skills",                       1, 1, 0),
    "0030302129": ("Military Sciences",                         1, 1, 0),
    "0040302211": ("Professional Practice",                     3, 4, 0),
    "0040302231": ("Entrepreneurship Bootcamp",                 4, 8, 0),
    "0030302232": ("Leadership Camp",                           1, 3, 0),  # BSc only

    # ── University Electives (1 credit each – pick from pool) ─────
    # BSc needs 3 × 1h;  Technical needs 1 × 1h
    # Using a representative subset of the 25+ options in the PDF
    "0030301222": ("Research and Technical Writing",            1, 3, 0),
    "0030302121": ("Science and Society Seminar I",             1, 3, 0),
    "0030302122": ("Arab Contributions to Science and Art",     1, 3, 0),
    "0030302133": ("Jerusalem: History and Civilization",       1, 3, 0),
    "0030302134": ("Jordan: History and Civilization",          1, 3, 0),
    "0040302221": ("Speech and Debate",                         1, 2, 0),
    "0000302222": ("Intro to Philosophy and Critical Thinking", 1, 2, 0),
    "0030302237": ("Free Choice Elective I",                    1, 0, 0),

    # ── College Compulsory (shared by all 6 plans) ─────────────────
    "0040303130": ("Fundamentals of Computing",                 4, 6, 0),
    "0030303111": ("Functional Math",                           3, 3, 0),
    "0030303121": ("STEM Lab I",                                1, 0, 3),
    "0040303121": ("Maths for Computing",                       3, 4, 0),
    "0040201100": ("Programming",                               3, 4, 0),
    "0040201290": ("Computing Project Planning",                4, 4, 0),
    "0040303221": ("Discrete Maths",                            3, 4, 0),

    # ── CS Major Courses ───────────────────────────────────────────
    "0040201200": ("Advanced Programming",                      3, 4, 0),
    "0040201201": ("Data Structures and Algorithms",            3, 4, 0),
    "0040201220": ("Software Development Lifecycles",           3, 4, 0),
    "0040201260": ("Website Design and Development",            3, 4, 0),
    "0040201261": ("Prototyping",                               3, 4, 0),
    "0040201321": ("Systems Analysis and Design",               3, 4, 0),
    "0040201341": ("Operating Systems",                         3, 4, 0),
    "0040201360": ("Application Development",                   3, 4, 0),
    "0000201391": ("Computing Research Project",                6, 6, 0),
    "0010203180": ("Networking",                                3, 4, 0),
    "0000203280": ("Security",                                  3, 4, 0),
    "0010203380": ("Computer Organization and Design",          3, 4, 0),
    "0010204282": ("Database Design and Development",           3, 4, 0),
    "0000204313": ("Business Process Support",                  3, 4, 0),
    # BSc-only CS major courses
    "0040201320": ("ERP Systems",                               3, 4, 0),
    "0040201362": ("Games Engine and Scripting",                3, 4, 0),
    "0040201430": ("Database Programming",                      3, 4, 0),
    "0040201440": ("Systems Programming",                       3, 4, 0),
    "0040201491": ("Capstone Project I",                        1, 0, 0),
    "0040201492": ("Capstone Project II",                       2, 0, 0),

    # CS Major Electives (BSc only, pick 3 × 3h from pool)
    "0040201441": ("Internet of Things",                        3, 4, 0),
    "0040201442": ("Real Time Systems",                         3, 4, 0),
    "0040201450": ("Cloud Computing",                           3, 4, 0),
    "0040201460": ("E-Commerce",                                3, 4, 0),
    "0040201461": ("Mobile Application Development",            3, 4, 0),
    "0040201462": ("Virtual and Augmented Reality Development", 3, 4, 0),

    # ── AI Major Courses ───────────────────────────────────────────
    # (shares several codes with CS above: 0040201201, 0040201220,
    #  0040201341, 0000201391, 0010203180, 0000203280, 0010204282,
    #  0000204313)
    "0010204210": ("Data Analytics",                            3, 4, 0),
    "0010204250": ("AI and Intelligent Systems",                3, 4, 0),
    "0000204280": ("Principles of Data Science and Computing",  3, 4, 0),
    "0010204281": ("Data Science Programming",                  3, 4, 0),
    "0000204310": ("Advanced Programming for Data Analysis",    3, 4, 0),  # hidden prereq
    "0000204311": ("Big Data Analytics and Visualization",      3, 4, 0),
    "0010204330": ("Modeling and Simulation",                   3, 4, 0),
    "0010204350": ("Machine Learning",                          3, 4, 0),
    "0010204351": ("Natural Language Processing",               3, 4, 0),
    "0000204430": ("Data Mining",                               3, 4, 0),
    "0010204450": ("Deep Learning",                             3, 4, 0),
    # BSc-only AI major courses
    "0010204491": ("Capstone Project I (AI)",                   1, 0, 0),
    "0010204492": ("Capstone Project II (AI)",                  2, 0, 0),

    # AI Major Electives (BSc only, pick 3 × 3h)
    "0000204410": ("Health Informatics",                        3, 4, 0),
    "0000204411": ("Time Series Analysis",                      3, 4, 0),
    "0000204412": ("Applied Analytical Models",                 3, 4, 0),
    "0010204431": ("Information Retrieval",                     3, 4, 0),
    "0010204451": ("Pattern Recognition",                       3, 4, 0),
    "0010204452": ("Computer Vision",                           3, 4, 0),

    # ── Cyber Major Courses ────────────────────────────────────────
    # (shares: 0040201201, 0040201220, 0040201260, 0040201341,
    #  0000201391, 0010203180, 0000203280, 0010203380,
    #  0010204282, 0000204313)
    "0010203210": ("Network Security",                          3, 4, 0),
    "0000203360": ("Penetration Testing",                       3, 4, 0),
    "0010203361": ("Forensics",                                 3, 4, 0),
    "0010203362": ("Ethical Hacking",                           3, 4, 0),
    # BSc-only Cyber major courses
    "0010203300": ("Information Security Management",           3, 4, 0),
    "0010203340": ("Cryptography",                              3, 4, 0),
    "0000203400": ("Risk Analysis and Systems Testing",         3, 4, 0),
    "0000203420": ("Secure Coding",                             3, 4, 0),
    "0010203491": ("Capstone Project I (Cyber)",                1, 0, 0),
    "0010203492": ("Capstone Project II (Cyber)",               2, 0, 0),

    # Cyber Major Electives (BSc only, pick 3 × 3h)
    "0010203401": ("Incident Response Management",              3, 4, 0),
    "0010203410": ("Mobile and Wireless Security",              3, 4, 0),
    "0010203411": ("Internet of Things Security",               3, 4, 0),
    "0010203420": ("Secure System Design and Development",      3, 4, 0),
    "0010203421": ("Web Security",                              3, 4, 0),
    "0010204250_cy": ("AI and Intelligent Systems (Cyber Elec)", 3, 4, 0),  # alias

    # ── Apprenticeship Courses (Market Requirements, all plans) ───
    # CS plans
    "0000201291": ("Apprenticeship for Computer Science 1",     6, 0, 6),
    "0000201292": ("Apprenticeship for Computer Science 2",     6, 0, 6),
    "0000201392": ("Apprenticeship for Computer Science 3",     6, 0, 6),
    # AI plans
    "0000204290": ("Apprenticeship for DS and AI 1",            6, 0, 6),
    "0000204291": ("Apprenticeship for DS and AI 2",            6, 0, 6),
    "0000204391": ("Apprenticeship for DS and AI 3",            6, 0, 6),
    # Cyber plans
    "0000203290": ("Apprenticeship for Cybersecurity 1",        6, 0, 6),
    "0000203291": ("Apprenticeship for Cybersecurity 2",        6, 0, 6),
    "0000203391": ("Apprenticeship for Cybersecurity 3",        6, 0, 6),
}

# ── A2.  UNIVERSITY ELECTIVE POOL (same for all plans) ───────
UNI_ELEC_POOL = [
    "0030301222", "0030302121", "0030302122",
    "0030302133", "0030302134", "0040302221",
    "0000302222", "0030302237",
]



### ── A3.  PER-PLAN DEFINITIONS ─────────────────────────────────
Each plan is a dict with:

  total_hrs       – graduation credit threshold

  uni_comp        – university compulsory course codes

  uni_elec_count  – how many 1h electives the student must pick

  remedials       – always the same 3 codes

  college         – college compulsory codes

  major_comp      – major compulsory codes (counted for graduation)

  major_hidden    – hidden prereq codes (NOT counted for graduation)

  major_elec_pool – pool to pick major electives from (BSc only)

  major_elec_count– number of major electives to pick

  major_elec_hrs  – credit hours each major elective

  app1, app2, app3– apprenticeship course codes

  capstone_I      – capstone I code (or None for Technical)

  capstone_II     – capstone II code (or None for Technical)

  category_map    – code→category string for CSV output



In [3]:
# ── A3.  PER-PLAN DEFINITIONS ─────────────────────────────────
# Each plan is a dict with:
#   total_hrs       – graduation credit threshold
#   uni_comp        – university compulsory course codes  
#   uni_elec_count  – how many 1h electives the student must pick
#   remedials       – always the same 3 codes
#   college         – college compulsory codes
#   major_comp      – major compulsory codes (counted for graduation)
#   major_hidden    – hidden prereq codes (NOT counted for graduation)
#   major_elec_pool – pool to pick major electives from (BSc only)
#   major_elec_count– number of major electives to pick
#   major_elec_hrs  – credit hours each major elective
#   app1, app2, app3– apprenticeship course codes
#   capstone_I      – capstone I code (or None for Technical)
#   capstone_II     – capstone II code (or None for Technical)
#   category_map    – code→category string for CSV output

PLANS = {}

# ── Helper: category lookup
def _cat(code, compulsory_set, hidden_set, elec_set, app_set, remset):
    if code in remset:        return "University Requirement (Remedial)"
    if code in hidden_set:    return "Hidden Prerequisite"
    if code in app_set:       return "Market Requirement"
    if code in elec_set:      return "University Elective" if code in UNI_ELEC_POOL else "Major Elective"
    if code in compulsory_set:
        k = code[:4]
        if k in ("0030","0040") and int(code[4:6]) in range(30,32): return "University Compulsory"
        if k in ("0030","0040") and code[4:8] == "3030":           return "College Compulsory"
        return "Major Compulsory"
    return "Major Elective"

# Shared structures
_REMEDIALS   = ["0030301110", "0030301120", "0030303110"]
_UNI_COMP_BSC  = ["0030301111","0030301121","0030301122","0030301123","0030301124",
                   "0040302111","0030302129","0040302211","0040302231","0030302232"]
_UNI_COMP_TECH = ["0030301111","0030301121","0030301122","0030301123",
                   "0040302111","0030302129","0040302211","0040302231"]
_COLLEGE     = ["0040303130","0030303111","0030303121","0040303121",
                "0040201100","0040201290","0040303221"]

# CS shared major compulsory (both BSc and Tech)
_CS_MAJOR_SHARED = [
    "0040201200","0040201201","0040201220","0040201260","0040201261",
    "0040201321","0040201341","0040201360","0000201391",
    "0010203180","0000203280","0010203380","0010204282","0000204313",
]
# CS BSc-only major courses
_CS_MAJOR_BSC_ONLY = [
    "0040201320","0040201362","0040201430","0040201440",
    "0040201491","0040201492",
]
_CS_ELEC_POOL = ["0040201441","0040201442","0040201450",
                  "0040201460","0040201461","0040201462"]

# AI shared major compulsory
_AI_MAJOR_SHARED = [
    "0040201201","0040201220","0040201341","0000201391",
    "0010203180","0000203280",
    "0010204210","0010204250","0000204280","0010204281",
    "0010204282","0000204311","0000204313","0010204350",
]
# AI BSc-only
_AI_MAJOR_BSC_ONLY = [
    "0010204330","0010204351","0000204430","0010204450",
    "0010204491","0010204492",
]
_AI_HIDDEN   = ["0000204310"]  # prereq for Big Data, not in graduation count
_AI_ELEC_POOL = ["0000204410","0000204411","0000204412",
                  "0010204431","0010204451","0010204452"]

# Cyber shared major compulsory
_CY_MAJOR_SHARED = [
    "0040201201","0040201220","0040201260","0040201341","0000201391",
    "0010203180","0010203210","0000203280",
    "0000203360","0010203361","0010203362","0010203380",
    "0010204282","0000204313",
]
# Cyber BSc-only
_CY_MAJOR_BSC_ONLY = [
    "0010203300","0010203340","0000203400","0000203420",
    "0010203491","0010203492",
]
_CY_ELEC_POOL = ["0010203401","0010203410","0010203411",
                  "0010203420","0010203421"]


### Assemble the 6 PLANS dicts
--------------------------------------------------------------


In [4]:

def _make_plan(total_hrs, uni_comp, uni_elec_count, college,
               major_comp, major_hidden, elec_pool, elec_count,
               app1, app2, app3, cap1, cap2):
    return {
        "total_hrs":        total_hrs,
        "uni_comp":         uni_comp,
        "uni_elec_count":   uni_elec_count,
        "remedials":        _REMEDIALS[:],
        "college":          college,
        "major_comp":       major_comp,
        "major_hidden":     major_hidden,
        "elec_pool":        elec_pool,
        "elec_count":       elec_count,
        "app1": app1, "app2": app2, "app3": app3,
        "cap1": cap1, "cap2": cap2,
    }

PLANS["CS_Bachelor"] = _make_plan(
    135, _UNI_COMP_BSC, 3, _COLLEGE,
    _CS_MAJOR_SHARED + _CS_MAJOR_BSC_ONLY, [],
    _CS_ELEC_POOL, 3,
    "0000201291","0000201292","0000201392",
    "0040201491","0040201492",
)
PLANS["CS_Technical"] = _make_plan(
    105, _UNI_COMP_TECH, 1, _COLLEGE,
    _CS_MAJOR_SHARED, [],
    [], 0,
    "0000201291","0000201292","0000201392",
    None, None,
)
PLANS["AI_Bachelor"] = _make_plan(
    135, _UNI_COMP_BSC, 3, _COLLEGE,
    _AI_MAJOR_SHARED + _AI_MAJOR_BSC_ONLY, _AI_HIDDEN,
    _AI_ELEC_POOL, 3,
    "0000204290","0000204291","0000204391",
    "0010204491","0010204492",
)
PLANS["AI_Technical"] = _make_plan(
    105, _UNI_COMP_TECH, 1, _COLLEGE,
    _AI_MAJOR_SHARED, _AI_HIDDEN,
    [], 0,
    "0000204290","0000204291","0000204391",
    None, None,
)
PLANS["Cyber_Bachelor"] = _make_plan(
    135, _UNI_COMP_BSC, 3, _COLLEGE,
    _CY_MAJOR_SHARED + _CY_MAJOR_BSC_ONLY, [],
    _CY_ELEC_POOL, 3,
    "0000203290","0000203291","0000203391",
    "0010203491","0010203492",
)
PLANS["Cyber_Technical"] = _make_plan(
    105, _UNI_COMP_TECH, 1, _COLLEGE,
    _CY_MAJOR_SHARED, [],
    [], 0,
    "0000203290","0000203291","0000203391",
    None, None,
)



### ── A4.  PREREQUISITE CHAINS ──────────────────────────────────
Format per entry: (course_code, prereq_code, allow_concurrent)

allow_concurrent = True  → co-requisite (can take in same semester)

allow_concurrent = False → hard prereq (must pass before registering)

These are SHARED across all plans that include the course.

Plan-specific overrides are handled separately (Cyber Tech vs BSc differ on the Ethical Hacking prereq).


In [5]:

# ── A4.  PREREQUISITE CHAINS ──────────────────────────────────
# Format per entry: (course_code, prereq_code, allow_concurrent)
# allow_concurrent = True  → co-requisite (can take in same semester)
# allow_concurrent = False → hard prereq (must pass before registering)
#
# These are SHARED across all plans that include the course.
# Plan-specific overrides are handled separately (Cyber Tech vs BSc
# differ on the Ethical Hacking prereq).

# University prereqs (same for every plan that includes the course)
_UNI_PREREQS = [
    ("0030301111", "0030301110", False),  # Arabic Skills → Remedial Arabic
    ("0030301121", "0030301120", False),  # Eng Pre-Int → Remedial English
    ("0030301122", "0030301121", False),  # Eng Int → Eng Pre-Int
    ("0030301123", "0030301122", False),  # Eng Upper → Eng Int
    ("0030301124", "0030301123", False),  # Eng Adv → Eng Upper  (BSc only)
    ("0040302211", "0040302111", False),  # Prof Practice → Prof Skills
    ("0040302211", "0030301122", False),  # Prof Practice → Eng Int
    ("0040302231", "0040302211", False),  # Entrepreneurship → Prof Practice
    ("0040302231", "0030301123", False),  # Entrepreneurship → Eng Upper
    ("0030302232", "0040302231", False),  # Leadership → Entrepreneurship (BSc)
]

# College prereqs (same for every plan)
_COLLEGE_PREREQS = [
    ("0040201100", "0040303130", False),  # Programming → Fund Comp
    ("0040201290", "0040302211", False),  # Project Planning → Prof Practice
    ("0030303111", "0030303110", False),  # Functional Math → Remedial Math
    ("0040303121", "0030303111", False),  # Maths for Computing → Func Math
    # Discrete Maths: hard prereq = Maths for Computing,  co-req = DSA
    ("0040303221", "0040303121", False),  # Discrete → Maths for Computing
    ("0040303221", "0040201201", True),   # Discrete co-req DSA   (any plan with both)
]

# CS major prereqs
_CS_MAJOR_PREREQS = [
    ("0040201200", "0040201100", False),  # Adv Prog → Programming
    ("0040201201", "0040201100", False),  # DSA → Programming
    ("0040201220", "0040201100", False),  # SDLC → Programming
    ("0040201260", "0040201100", False),  # Web Design → Programming
    ("0040201261", "0040201220", False),  # Prototyping → SDLC
    ("0040201321", "0040201220", False),  # Sys Analysis → SDLC
    ("0040201341", "0040201201", False),  # OS → DSA
    ("0040201360", "0040201261", False),  # App Dev → Prototyping
    ("0000201391", "0040201290", False),  # CRP → Project Planning
    ("0010203180", "0040303130", False),  # Networking → Fund Comp
    ("0000203280", "0040303130", False),  # Security → Fund Comp
    # Comp Org: hard prereq = Programming, co-req = Discrete Maths
    ("0010203380", "0040201100", False),
    ("0010203380", "0040303221", True),   # co-req Discrete Maths
    ("0010204282", "0040201100", False),  # DB Design → Programming
    ("0000204313", "0010204282", False),  # Bus Process → DB Design
    # BSc-only
    ("0040201320", "0040201360", False),  # ERP → App Dev
    ("0040201362", "0040201100", False),  # Games Engine → Programming
    ("0040201430", "0010204282", False),  # DB Programming → DB Design
    ("0040201430", "0040201201", False),  # DB Programming → DSA
    ("0040201440", "0040201341", False),  # Sys Programming → OS
    ("0040201491", "0000201391", False),  # Capstone I → CRP
    ("0040201492", "0040201491", False),  # Capstone II → Capstone I
    # CS Apprenticeship chain
    ("0000201292", "0000201291", False),  # App2 → App1
    ("0000201392", "0000201292", False),  # App3 → App2 (after the double-sem)
]

# AI major prereqs
_AI_MAJOR_PREREQS = [
    ("0040201201", "0040201100", False),
    ("0040201220", "0040201100", False),
    ("0040201341", "0040201201", False),
    ("0000201391", "0040201290", False),
    ("0010203180", "0040303130", False),
    ("0000203280", "0040303130", False),
    ("0010204210", "0040303130", False),  # Data Analytics → Fund Comp
    ("0010204250", "0040303121", False),  # AI Systems → Maths for Computing
    ("0000204280", "0040303121", False),  # DS Principles → Maths for Computing
    ("0010204281", "0040201100", False),  # DS Programming → Programming
    ("0010204282", "0040201100", False),
    # Hidden prereq chain: Adv Prog for Data Analysis → Big Data
    ("0000204310", "0040201100", False),  # Hidden: Adv Prog → Programming
    ("0000204311", "0000204310", False),  # Big Data → Adv Prog for Data Analysis
    ("0000204313", "0010204282", False),
    # ML needs Data Analytics AND DS Principles
    ("0010204350", "0010204210", False),
    ("0010204350", "0000204280", False),
    # BSc-only AI
    ("0010204330", "0010204250", False),  # Modeling → AI Systems
    ("0010204351", "0010204250", False),  # NLP → AI Systems
    ("0000204430", "0010204210", False),  # Data Mining → Data Analytics
    ("0010204450", "0010204350", False),  # Deep Learning → ML
    ("0010204491", "0000201391", False),  # Capstone I → CRP
    ("0010204492", "0010204491", False),
    # AI Apprenticeship chain
    ("0000204291", "0000204290", False),
    ("0000204391", "0000204291", False),
]

# Cyber major prereqs (BSc version – Ethical Hacking only needs Security)
_CY_MAJOR_PREREQS_BSC = [
    ("0040201201", "0040201100", False),
    ("0040201220", "0040201100", False),
    ("0040201260", "0040201100", False),
    ("0040201341", "0040201201", False),
    ("0000201391", "0040201290", False),
    ("0010203180", "0040303130", False),
    ("0010203210", "0000203280", False),  # Net Security → Security
    ("0000203280", "0040303130", False),
    # Ethical Hacking → Security (BSc rule from PDF)
    ("0010203362", "0000203280", False),
    ("0000203360", "0010203362", False),  # Pen Testing → Ethical Hacking
    # Forensics → Comp Org AND Security
    ("0010203361", "0010203380", False),
    ("0010203361", "0000203280", False),
    # Comp Org: hard=Programming, co-req=Discrete Maths
    ("0010203380", "0040201100", False),
    ("0010203380", "0040303221", True),
    ("0010204282", "0040201100", False),
    ("0000204313", "0010204282", False),
    # BSc-only
    ("0010203300", "0010203210", False),  # Info Sec Mgmt → Net Security
    ("0010203340", "0040303221", False),  # Cryptography → Discrete Maths
    ("0000203400", "0000203280", False),  # Risk Analysis → Security
    ("0000203420", "0010204282", False),  # Secure Coding → DB Design
    ("0010203491", "0000201391", False),  # Capstone I → CRP
    ("0010203492", "0010203491", False),
    # Cyber Apprenticeship chain
    ("0000203291", "0000203290", False),
    ("0000203391", "0000203291", False),
]

# Cyber Technical: Ethical Hacking requires Network Security (not just Security)
_CY_MAJOR_PREREQS_TECH = [r for r in _CY_MAJOR_PREREQS_BSC
                            if not (r[0] == "0010203362" and r[1] == "0000203280")]
_CY_MAJOR_PREREQS_TECH.append(("0010203362", "0010203210", False))  # Ethical Hack → Net Sec (Tech)

# Full prereq list per plan
PLAN_PREREQS = {
    "CS_Bachelor":   _UNI_PREREQS + _COLLEGE_PREREQS + _CS_MAJOR_PREREQS,
    "CS_Technical":  _UNI_PREREQS + _COLLEGE_PREREQS + _CS_MAJOR_PREREQS,
    "AI_Bachelor":   _UNI_PREREQS + _COLLEGE_PREREQS + _AI_MAJOR_PREREQS,
    "AI_Technical":  _UNI_PREREQS + _COLLEGE_PREREQS + _AI_MAJOR_PREREQS,
    "Cyber_Bachelor": _UNI_PREREQS + _COLLEGE_PREREQS + _CY_MAJOR_PREREQS_BSC,
    "Cyber_Technical": _UNI_PREREQS + _COLLEGE_PREREQS + _CY_MAJOR_PREREQS_TECH,
}




==============================================================
## PART B  —  CSV GENERATOR
==============================================================


In [6]:

def get_category(code, plan_key):
    """Return the proper category string for a course code in a given plan."""
    p = PLANS[plan_key]
    if code in _REMEDIALS:             return "University Requirement (Remedial)"
    if code in UNI_ELEC_POOL:          return "University Elective"
    if code in p["uni_comp"]:          return "University Compulsory"
    if code in _COLLEGE:               return "College Compulsory"
    if code in [p["app1"],p["app2"],p["app3"]]: return "Market Requirement"
    if code in p["major_hidden"]:      return "Hidden Prerequisite"
    if code in p["major_comp"]:        return "Major Compulsory"
    if code in p["elec_pool"]:         return "Major Elective"
    return "Major Elective"


def build_courses_df(plan_key):
    """
    Build a DataFrame of all courses in the plan, ready to write as CSV.
    Includes the 'Hidden Prerequisite' courses (needed for prereq chain
    but not counted toward graduation hours).
    """
    p = PLANS[plan_key]
    is_bsc = "Bachelor" in plan_key
    rows = []

    all_codes = (
        _REMEDIALS
        + p["uni_comp"]
        + UNI_ELEC_POOL                          # full pool (student picks subset)
        + _COLLEGE
        + [p["app1"], p["app2"], p["app3"]]
        + p["major_hidden"]
        + p["major_comp"]
        + p["elec_pool"]
    )
    seen = set()
    for code in all_codes:
        if code in seen:
            continue
        seen.add(code)
        entry = CATALOG.get(code)
        if entry is None:
            continue
        name, ch, lh, lab = entry
        rows.append({
            "course_code":   code,
            "course_name_en": name,
            "credit_hours":   ch,
            "lecture_hours":  lh,
            "lab_hours":      lab,
            "category":       get_category(code, plan_key),
            "plan_source":    plan_key.replace("_", "-"),
        })
    return pd.DataFrame(rows)


def build_prereqs_df(plan_key):
    """
    Build a DataFrame of prerequisite rules for the plan.
    Filters out rules whose courses are not in this plan.
    """
    p      = PLANS[plan_key]
    raw    = PLAN_PREREQS[plan_key]
    # Collect all codes present in this plan
    valid  = (
        set(_REMEDIALS) | set(p["uni_comp"]) | set(UNI_ELEC_POOL)
        | set(_COLLEGE)
        | {p["app1"], p["app2"], p["app3"]}
        | set(p["major_hidden"])
        | set(p["major_comp"])
        | set(p["elec_pool"])
    )
    if p["cap1"]: valid.add(p["cap1"])
    if p["cap2"]: valid.add(p["cap2"])

    rows = []
    seen = set()
    for (cc, pc, conc) in raw:
        if cc in valid and pc in valid:
            key = (cc, pc)
            if key not in seen:
                seen.add(key)
                rows.append({
                    "course_code":      cc,
                    "prereq_code":      pc,
                    "allow_concurrent": conc,
                })
    return pd.DataFrame(rows)


def generate_all_csvs(out_dir=OUT_DIR):
    """Write all 12 CSV files (6 course files + 6 prereq files)."""
    os.makedirs(out_dir, exist_ok=True)
    for plan_key in PLANS:
        major, deg = plan_key.split("_")
        prefix = f"{major}_{deg}"
        cf = os.path.join(out_dir, f"{prefix}_Courses.csv")
        pf = os.path.join(out_dir, f"{prefix}_Prerequisites.csv")
        build_courses_df(plan_key).to_csv(cf, index=False)
        build_prereqs_df(plan_key).to_csv(pf, index=False)
        print(f"  Wrote {cf.split(os.sep)[-1]}")
        print(f"  Wrote {pf.split(os.sep)[-1]}")




==============================================================
## PART C  —  SIMULATION ENGINE
==============================================================


In [ ]:
# ── Grade scale ───────────────────────────────────────────────
GRADE_SYMBOLS  = ["D",   "M",   "P",   "U",   "W"]
GRADE_POINTS   = {"D":4.0, "M":3.2, "P":2.4, "U":1.6, "W":0.0}
IS_PASSING     = {"D":True,"M":True,"P":True,"U":False,"W":False}
GPA_INCLUDED   = {"D":True,"M":True,"P":True,"U":True, "W":False}
# Realistic grade distribution (heavy toward passing)
GRADE_PROBS    = [0.17, 0.33, 0.43, 0.05, 0.02]

# ── Simulation parameters ─────────────────────────────────────
NUM_STUDENTS        = 3000
ADMISSION_YEARS     = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
ADM_YEAR_WEIGHTS    = [0.10, 0.12, 0.15, 0.18, 0.18, 0.15, 0.12]
SIM_END_YEAR        = 2025
SIM_END_TERM        = "Fall"

# Probability each student needs each independent remedial course
PROB_REM_ARABIC  = 0.18
PROB_REM_ENGLISH = 0.25
PROB_REM_MATH    = 0.15

# Summer semester skip probabilities
SUMMER_SKIP_CLEAN = 0.55   # skip summer if no failing courses
SUMMER_SKIP_FAIL  = 0.20   # skip summer if has failing courses (catch-up mode)

# Regular semester credit load: 12-18h (weighted toward 14-16h)
REG_CREDIT_CHOICES  = list(range(12, 19))
REG_CREDIT_WEIGHTS  = [0.05, 0.10, 0.20, 0.30, 0.25, 0.07, 0.03]

# Trigger apprenticeship once student has passed this many
# non-app, non-remedial, non-hidden hours
APP_TRIGGER_HRS = 48



# ── C1.  Semester sequence generator ─────────────────────────
def make_semester_sequence(start_year,
                           end_year=SIM_END_YEAR,
                           end_term=SIM_END_TERM):
    """
    Yield (calendar_year, term) tuples in correct HTU academic order:
        Fall(Y) → Spring(Y) → Summer(Y) → Fall(Y+1) → …

    Note: 'Spring' and 'Summer' belong to the calendar year AFTER Fall.
    but even so it still counts as the same academic 
    year so it counts as the same year as fall. 
    This matches how HTU labels its semesters (Fall 2022 is the start of
    the 2022-23 academic year; and for Spring even if it's in 2023 it is written
    as 2022 also).
    """
    ORDER = {"Fall": 1, "Spring": 2, "Summer": 3}
    stop_key = end_year * 10 + ORDER[end_term]
    y = start_year
    while True:
        for (cy, term) in [(y, "Fall"), (y, "Spring"), (y, "Summer")]:
            if cy * 10 + ORDER[term] > stop_key:
                return
            yield (cy, term)
        y += 1


# ── C2.  Per-student plan initializer ────────────────────────
def init_student_plan(plan_key, adm_year,
                      needs_rem_arabic, needs_rem_english, needs_rem_math):
    """
    Build the simulation state for a single student:
      - full_plan_codes : every code the student must attempt
      - grad_required   : codes that must be passed to graduate
      - prereq_map      : code → [(prereq_code, allow_concurrent), …]
      - credit_map      : code → credit_hours
      - name_map        : code → English name
    Also picks the student's specific elective assignments (university
    and major electives are randomly drawn from their pools here, so each
    student has a fixed personal curriculum).
    """
    p = PLANS[plan_key]

    # ── Assign electives for THIS student ──
    # University electives (1h each)
    uni_elecs = random.sample(UNI_ELEC_POOL, p["uni_elec_count"])

    # Major electives (3h each, BSc only)
    maj_elecs = (random.sample(p["elec_pool"], p["elec_count"])
                 if p["elec_count"] > 0 else [])

    # ── Build full plan ──
    # All courses this student personally needs to pass
    personal_plan = (
        p["uni_comp"] + uni_elecs + _COLLEGE
        + [p["app1"], p["app2"], p["app3"]]
        + p["major_hidden"]          # must pass for prereq chain, not counted
        + p["major_comp"]
        + maj_elecs
    )

    # Add remedials only for the ones the student actually needs
    remedials_needed = []
    if needs_rem_arabic:  remedials_needed.append("0030301110")
    if needs_rem_english: remedials_needed.append("0030301120")
    if needs_rem_math:    remedials_needed.append("0030303110")

    # Courses required to graduate (excludes remedials, hidden prereqs)
    grad_required = set(
        p["uni_comp"] + uni_elecs + _COLLEGE
        + [p["app1"], p["app2"], p["app3"]]
        + p["major_comp"] + maj_elecs
    )

    # Students who do NOT need a specific remedial start with it "passed"
    # (they passed the cognitive test = exempt)
    pre_passed = set()
    if not needs_rem_arabic:  pre_passed.add("0030301110")
    if not needs_rem_english: pre_passed.add("0030301120")
    if not needs_rem_math:    pre_passed.add("0030303110")

    # ── Build prereq map ──
    # Filter PLAN_PREREQS to only rules relevant to this student's plan
    full_codes = set(personal_plan) | set(remedials_needed) | pre_passed
    prereq_map = {}   # course_code → [(prereq_code, allow_concurrent)]
    for (cc, pc, conc) in PLAN_PREREQS[plan_key]:
        if cc in full_codes:
            prereq_map.setdefault(cc, []).append((pc, conc))

    return {
        "full_plan":     remedials_needed + personal_plan,
        "grad_required": grad_required,
        "pre_passed":    pre_passed,
        "prereq_map":    prereq_map,
        "credit_map":    {c: CATALOG[c][1] for c in (remedials_needed + personal_plan) if c in CATALOG},
        "name_map":      {c: CATALOG[c][0] for c in (remedials_needed + personal_plan) if c in CATALOG},
        "app1": p["app1"], "app2": p["app2"], "app3": p["app3"],
        "cap1": p["cap1"], "cap2": p["cap2"],
        "total_hrs": p["total_hrs"],
        "major_hidden": set(p["major_hidden"]),
    }


# ── C3.  Eligibility checker ──────────────────────────────────
def get_eligible(plan_state, passed):
    """
    Two-pass eligibility check:
      Pass 1 → hard prereqs (allow_concurrent=False) all in `passed`
      Pass 2 → co-reqs (allow_concurrent=True) in `passed` OR in pass-1 set
    Returns a set of codes the student CAN register this semester,
    excluding anything already passed.
    """
    prereq_map  = plan_state["prereq_map"]
    all_codes   = set(plan_state["full_plan"])

    # Pass 1: all hard prereqs satisfied
    pass1 = set()
    for c in all_codes:
        if c in passed:
            continue
        hard_reqs = [pc for (pc, conc) in prereq_map.get(c, []) if not conc]
        if all(r in passed for r in hard_reqs):
            pass1.add(c)

    # Pass 2: all co-reqs in passed OR in pass-1 set
    eligible = set()
    for c in pass1:
        coreqs = [pc for (pc, conc) in prereq_map.get(c, []) if conc]
        if all(r in passed or r in pass1 for r in coreqs):
            eligible.add(c)

    return eligible


# ── C4.  Course selector ──────────────────────────────────────
def select_courses(eligible, fail_retry, credit_limit, credit_map):
    """
    Greedy credit-respecting selection:
      1. Failed/withdrawn courses first (retry priority)
      2. Then remaining eligible courses in order
    Zero-credit courses (remedials) always fit – they don't consume
    the credit budget.
    """
    ordered = (
        [c for c in fail_retry if c in eligible]
        + [c for c in eligible if c not in fail_retry]
    )
    selected, used = [], 0
    for c in ordered:
        ch = credit_map.get(c, 3)
        if ch == 0:                       # remedial: free slot
            selected.append(c)
        elif used + ch <= credit_limit:
            selected.append(c)
            used += ch
    return selected


# ── C5.  GPA helpers ─────────────────────────────────────────
def sem_gpa(records):
    """Semester GPA for a list of enrollment records."""
    pts = sum(GRADE_POINTS[r["grade"]] * r["credit_hours"]
              for r in records
              if GPA_INCLUDED[r["grade"]] and r["credit_hours"] > 0)
    hrs = sum(r["credit_hours"] for r in records
              if GPA_INCLUDED[r["grade"]] and r["credit_hours"] > 0)
    return round(pts / hrs, 2) if hrs > 0 else 0.0


def cum_gpa(all_records):
    """
    Cumulative GPA using last-attempt-wins rule (W excluded).
    When a student retakes a failed course, the new grade replaces
    the old U in the cumulative calculation.
    """
    latest = {}
    for r in all_records:
        if r["grade"] == "W":
            continue
        cid = r["course_code"]
        # Overwrite with latest attempt (handles U→P replacement)
        if cid not in latest or not IS_PASSING[latest[cid]["grade"]]:
            latest[cid] = r
    pts = sum(GRADE_POINTS[v["grade"]] * v["credit_hours"]
              for v in latest.values() if v["credit_hours"] > 0)
    hrs = sum(v["credit_hours"] for v in latest.values() if v["credit_hours"] > 0)
    return round(pts / hrs, 2) if hrs > 0 else 0.0


# ── C6.  Main simulation loop ────────────────────────────────
def run_simulation(num_students=NUM_STUDENTS):
    """
    Simulate `num_students` students across all 6 plans.
    Returns (students_df, enrollments_df).
    """
    PLAN_KEYS = list(PLANS.keys())
    students_data    = []
    enrollments_data = []

    print(f"Simulating {num_students} students …")

    for s_idx in range(1, num_students + 1):
        # ── Assign student attributes ──────────────────────
        plan_key   = random.choice(PLAN_KEYS)
        adm_year   = random.choices(ADMISSION_YEARS, weights=ADM_YEAR_WEIGHTS)[0]
        student_id = f"20{adm_year % 100:02d}{s_idx:05d}"

        # Each remedial is decided INDEPENDENTLY
        needs_rem_ar  = random.random() < PROB_REM_ARABIC
        needs_rem_en  = random.random() < PROB_REM_ENGLISH
        needs_rem_ma  = random.random() < PROB_REM_MATH

        ps = init_student_plan(plan_key, adm_year,
                                needs_rem_ar, needs_rem_en, needs_rem_ma)

        # ── Per-student state ──────────────────────────────
        passed          = set(ps["pre_passed"])   # pre-passed = exempt remeds
        failed_count    = {}   # course_code → count of (U or W) outcomes
        all_history     = []   # all enrollment records (for cum GPA calc)
        cum_passed_hrs  = 0    # running total of passed credit hours
        status          = "Active"
        sem_number      = 0
        last_cum_gpa    = 0.0
        last_sem_gpa    = 0.0

        # Apprenticeship state:
        #   0 = not started
        #   1 = first semester (App1+App2) is done, need App3
        #   2 = complete
        app_stage = 0

        # ── Walk through semesters ─────────────────────────
        for (year, term) in make_semester_sequence(adm_year):
            if status == "Graduated":
                break
            is_summer = (term == "Summer")

            # ── Summer skip logic ──
            has_fails = any(v > 0 for v in failed_count.values())
            if is_summer:
                skip_p = SUMMER_SKIP_FAIL if has_fails else SUMMER_SKIP_CLEAN
                if random.random() < skip_p:
                    # Only skip if NOT in the middle of apprenticeship
                    # (during apprenticeship, we must not skip unless app_stage=0)
                    if app_stage == 0:
                        continue

            # ── Check graduation ──
            if ps["grad_required"].issubset(passed):
                status = "Graduated"
                break

            # ── Determine registration mode ────────────────
            eligible = get_eligible(ps, passed)

            # -- APPRENTICESHIP MODE --
            # Trigger: student has passed enough non-app hours
            # and hasn't started the block yet.
            non_app_passed_hrs = sum(
                CATALOG[c][1] for c in passed
                if c in CATALOG and c not in [ps["app1"], ps["app2"], ps["app3"]]
                and CATALOG[c][1] > 0
                and c not in _REMEDIALS
                and c not in ps["major_hidden"]
            )

            in_app_block = False
            if (app_stage == 0
                    and non_app_passed_hrs >= APP_TRIGGER_HRS
                    and ps["app1"] not in passed
                    and not is_summer):
                # START apprenticeship: first semester must be non-summer
                # (summer can't take the 2-app double block)
                app_stage = 1
                in_app_block = True
            elif app_stage == 1:
                in_app_block = True
            # app_stage == 2 → done; fall through to normal mode

            if in_app_block:
                if app_stage == 1 and not is_summer:
                    # Double semester: App1 + App2 (12h)
                    app_registered = []
                    if ps["app1"] not in passed and ps["app1"] in eligible:
                        app_registered.append(ps["app1"])
                    if ps["app2"] not in passed and ps["app2"] in eligible:
                        app_registered.append(ps["app2"])

                    # If App1 not yet eligible (edge case), just wait
                    if not app_registered:
                        continue

                    registered = app_registered

                    # Allow Capstone alongside apprenticeship (BSc only)
                    for cap in [ps["cap1"], ps["cap2"]]:
                        if cap and cap not in passed and cap in eligible:
                            registered.append(cap)

                    # After this semester: App1+App2 done
                    # app_stage becomes 2 once we detect App3 done,
                    # but set to indicate "double done, need single"
                    # We mark this semester completion below

                elif app_stage == 1 and is_summer:
                    # Summer but still on stage 1 (double not done yet)
                    # The double block started in a previous Fall/Spring
                    # and now it's summer. Take App3 in this summer.
                    # The user rule: summer = single course.
                    app_registered = []
                    if ps["app3"] not in passed and ps["app3"] in eligible:
                        app_registered.append(ps["app3"])
                    if not app_registered:
                        continue
                    registered = app_registered
                    for cap in [ps["cap1"], ps["cap2"]]:
                        if cap and cap not in passed and cap in eligible:
                            registered.append(cap)

                else:
                    # app_stage == 1 but logic fell through – shouldn't happen
                    registered = []
                    continue

            else:
                # ── NORMAL SEMESTER MODE ──────────────────
                if not eligible:
                    # Nothing to register → check graduation
                    if ps["grad_required"].issubset(passed):
                        status = "Graduated"
                    break

                # Compute effective credit limit
                remaining_hrs = sum(
                    CATALOG[c][1] for c in ps["grad_required"] if c not in passed
                    and c in CATALOG
                )
                if is_summer:
                    if remaining_hrs <= 12:
                        # Graduation exception in summer: up to 12h
                        credit_limit = min(remaining_hrs, 12)
                    else:
                        credit_limit = random.randint(3, 9)
                else:
                    if remaining_hrs <= 21:
                        # Graduation exception: take everything up to 21h
                        credit_limit = min(remaining_hrs, 21)
                    elif remaining_hrs < 12:
                        # Below minimum – just take what's left
                        credit_limit = remaining_hrs
                    else:
                        credit_limit = random.choices(
                            REG_CREDIT_CHOICES, weights=REG_CREDIT_WEIGHTS)[0]

                # Build retry list (failed/withdrawn courses have priority)
                fail_retry = [c for c in eligible if failed_count.get(c, 0) > 0]

                registered = select_courses(
                    eligible, fail_retry, credit_limit, ps["credit_map"])

                if not registered:
                    continue

            # ── Grade and record ───────────────────────────
            sem_number += 1
            sem_label   = f"{year}_{term}"
            sem_records = []

            for c in registered:
                ch       = ps["credit_map"].get(c, 0)
                cname    = ps["name_map"].get(c, c)
                grade    = random.choices(GRADE_SYMBOLS, weights=GRADE_PROBS)[0]
                is_retry = failed_count.get(c, 0) > 0

                rec = {
                    "student_id":            student_id,
                    "course_code":           c,
                    "course_name":           cname,
                    "credit_hours":          ch,
                    "semester_label":        sem_label,
                    "year":                  year,
                    "term":                  term,
                    "semester_number":       sem_number,
                    "grade":                 grade,
                    "is_retake":             is_retry,
                    "cum_passed_hrs_before": cum_passed_hrs,  # cumulative hours before this semester
                    # GPA snapshots appended below
                }
                sem_records.append(rec)
                all_history.append(rec)

                # Update state
                if IS_PASSING[grade]:
                    passed.add(c)
                    failed_count.pop(c, None)
                    # cum_passed_hrs += ch  # moved outside the loop
                elif grade in ("U", "W"):
                    failed_count[c] = failed_count.get(c, 0) + 1

            # Update cum_passed_hrs after all courses in the semester
            for rec in sem_records:
                if IS_PASSING[rec["grade"]]:
                    cum_passed_hrs += rec["credit_hours"]

            # ── Update apprenticeship stage ────────────────
            if in_app_block and app_stage == 1:
                # Check if both App1 and App2 were attempted
                attempted_this_sem = set(r["course_code"] for r in sem_records)
                double_attempted   = (ps["app1"] in attempted_this_sem
                                      or ps["app2"] in attempted_this_sem)
                single_attempted   = ps["app3"] in attempted_this_sem
                if double_attempted and not single_attempted:
                    # Double sem done; next sem will be single (App3)
                    app_stage = 1   # stays 1 until App3 is also done
                    # Re-label: next iteration app_stage=1 and is_summer checks
                    # handled above. Force stage to "waiting for App3":
                    # We use a trick: mark app_stage = 1.5 conceptually,
                    # but since we can't use floats, we look at whether
                    # App1+App2 are passed instead.
                if ps["app3"] in passed:
                    app_stage = 2   # Apprenticeship fully complete

            # ── Compute GPA snapshots ──────────────────────
            s_gpa = sem_gpa(sem_records)
            c_gpa = cum_gpa(all_history)
            last_sem_gpa, last_cum_gpa = s_gpa, c_gpa

            for rec in sem_records:
                rec["sem_gpa"] = s_gpa
                rec["cum_gpa"] = c_gpa

            enrollments_data.extend(sem_records)

            # ── Re-check graduation after this semester ────
            if ps["grad_required"].issubset(passed):
                status = "Graduated"
                break

        # ── Finalise student record ────────────────────────
        students_data.append({
            "student_id":         student_id,
            "plan_key":           plan_key,
            "major":              plan_key.split("_")[0],
            "degree_type":        plan_key.split("_")[1],
            "admission_year":     adm_year,
            "needs_rem_arabic":   needs_rem_ar,
            "needs_rem_english":  needs_rem_en,
            "needs_rem_math":     needs_rem_ma,
            "status":             status,
            "semesters_taken":    sem_number,
            "final_cum_gpa":      last_cum_gpa,
            "total_passed_hrs":   cum_passed_hrs,
        })

        if s_idx % 500 == 0:
            print(f"  … {s_idx}/{num_students} students processed")

    return pd.DataFrame(students_data), pd.DataFrame(enrollments_data)



==============================================================
## PART D  —  MAIN
==============================================================



In [ ]:

if __name__ == "__main__":
    print("=" * 60)
    print("HTU Data Generator")
    print("=" * 60)

    # ── Step 1: Generate CSV plan files ───────────────────────
    print("\n[1/2] Generating study plan CSV files …")
    generate_all_csvs(OUT_DIR)

    # ── Step 2: Validate credit totals ────────────────────────
    print("\n     Validating plan credit totals:")
    all_ok = True
    for plan_key, p in PLANS.items():
        df = build_courses_df(plan_key)
        active = df[~df["category"].isin(
            ["University Requirement (Remedial)", "Hidden Prerequisite",
             "University Elective", "Major Elective"])]
        # Count compulsory + pick 1/3 electives for the check
        comp_hrs = df[df["category"].isin(
            ["University Compulsory","College Compulsory","Market Requirement",
             "Major Compulsory"])]["credit_hours"].sum()
        elec_hrs = p["uni_elec_count"] * 1  # university electives
        melec_hrs = p["elec_count"] * 3      # major electives
        total = comp_hrs + elec_hrs + melec_hrs
        status_str = "✓" if total == p["total_hrs"] else f"✗ MISMATCH (expected {p['total_hrs']})"
        print(f"     {plan_key:<20}  {total:3d}h  {status_str}")
        if total != p["total_hrs"]:
            all_ok = False
    if not all_ok:
        print("\n  ⚠ Credit total mismatch detected – please review plan data.")
    else:
        print("     All plan totals correct ✓")

    # ── Step 3: Run simulation ─────────────────────────────────
    print(f"\n[2/2] Running student simulation ({NUM_STUDENTS} students) …")
    df_students, df_enrollments = run_simulation(NUM_STUDENTS)

    # ── Step 4: Save simulation output ────────────────────────
    stu_path = os.path.join(OUT_DIR, "htu_students.csv")
    enr_path = os.path.join(OUT_DIR, "htu_enrollments.csv")
    df_students.to_csv(stu_path, index=False)
    df_enrollments.to_csv(enr_path, index=False)

    # ── Step 5: Summary stats ──────────────────────────────────
    print("\n" + "=" * 60)
    print("GENERATION COMPLETE")
    print("=" * 60)
    print(f"  Students:          {len(df_students):,}")
    print(f"  Enrollment rows:   {len(df_enrollments):,}")
    print()
    print("  Student status:")
    for k, v in df_students["status"].value_counts().items():
        print(f"    {k:<12} {v:,}")
    print()
    print("  By major & degree:")
    grp = df_students.groupby(["major","degree_type"]).size().reset_index(name="n")
    for _, r in grp.iterrows():
        print(f"    {r['major']+' '+r['degree_type']:<22} {r['n']:,}")
    print()
    print("  Grade distribution:")
    gd = df_enrollments["grade"].value_counts(normalize=True).sort_index()
    for g, pct in gd.items():
        bar = "█" * int(pct * 40)
        print(f"    {g}  {bar}  {pct*100:.1f}%")
    print()
    print("  Avg semesters taken:")
    for deg, mean in df_students.groupby("degree_type")["semesters_taken"].mean().items():
        print(f"    {deg:<12} {mean:.1f}")
    print()
    print("  Top 10 courses by enrollment:")
    top10 = df_enrollments["course_name"].value_counts().head(10)
    for name, cnt in top10.items():
        print(f"    {cnt:6,}  {name}")
    print()
    print(f"  Files written to: {OUT_DIR}")
    print(f"    htu_students.csv")
    print(f"    htu_enrollments.csv")
    print(f"    (plus 12 plan CSV files)")


HTU Data Generator

[1/2] Generating study plan CSV files …
  Wrote CS_Bachelor_Courses.csv
  Wrote CS_Bachelor_Prerequisites.csv
  Wrote CS_Technical_Courses.csv
  Wrote CS_Technical_Prerequisites.csv
  Wrote AI_Bachelor_Courses.csv
  Wrote AI_Bachelor_Prerequisites.csv
  Wrote AI_Technical_Courses.csv
  Wrote AI_Technical_Prerequisites.csv
  Wrote Cyber_Bachelor_Courses.csv
  Wrote Cyber_Bachelor_Prerequisites.csv
  Wrote Cyber_Technical_Courses.csv
  Wrote Cyber_Technical_Prerequisites.csv

     Validating plan credit totals:
     CS_Bachelor           135h  ✓
     CS_Technical          105h  ✓
     AI_Bachelor           135h  ✓
     AI_Technical          105h  ✓
     Cyber_Bachelor        135h  ✓
     Cyber_Technical       105h  ✓
     All plan totals correct ✓

[2/2] Running student simulation (3000 students) …
Simulating 3000 students …
  … 500/3000 students processed
  … 1000/3000 students processed
  … 1500/3000 students processed
  … 2000/3000 students processed
  … 2500/3000 